# 99 Indonesian Speaker Corpus — ASR Evaluation

This notebook evaluates the Indonesian-adapted **Wav2Vec 2.0 XLS-R** checkpoint on the 99-speaker corpus.

For each speaker, the notebook:
1. Loads the full speaker recording and reference transcript.
2. Loads audio as mono at 16 kHz.
3. Runs the Hugging Face automatic-speech-recognition pipeline using 20-second chunks and a `(4, 2)` second stride.
4. Calculates speaker-level WER and CER, plus substitutions, deletions, insertions, and hits.
5. Saves a CSV checkpoint after each completed speaker.

The checkpoint used is `indonesian-nlp/wav2vec2-large-xlsr-indonesian`. The model is used for inference; no fine-tuning is performed in this notebook.

## 1. Environment and imports

Run this notebook in an environment with internet/model access for downloading the Hugging Face checkpoint, and access to the corpus files. CUDA is recommended for faster inference; CPU execution is supported.

In [ ]:
%pip install -q transformers librosa jiwer soundfile pandas

In [ ]:
import gc
import re
import time
from pathlib import Path

import librosa
import pandas as pd
import torch
from jiwer import wer, cer, process_words
from transformers import AutoProcessor, AutoModelForCTC, pipeline

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Dataset and model configuration

Set `DATASET_ROOT` to the folder containing `speaker01.wav` … `speaker99.wav` and `transcript01.txt` … `transcript99.txt`.

In [ ]:
DATASET_ROOT = Path(
    "/kaggle/input/datasets/mouchu/indonesian-speech-corpus-for-speaker-recognition"
)

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "wav2vec2_99speaker_results.csv"

MODEL_NAME = "indonesian-nlp/wav2vec2-large-xlsr-indonesian"
SAMPLE_RATE = 16000
CHUNK_LENGTH_S = 20
STRIDE_LENGTH_S = (4, 2)

print("Dataset directory exists:", DATASET_ROOT.exists())
print("Model checkpoint:", MODEL_NAME)
print("Results checkpoint:", CHECKPOINT_PATH)

## 3. Load processor, model, and ASR pipeline

The audio is resampled to 16 kHz and converted to mono before inference. Long recordings are processed by the ASR pipeline in 20-second chunks with 4 seconds of left stride and 2 seconds of right stride.

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForCTC.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=CHUNK_LENGTH_S,
    stride_length_s=STRIDE_LENGTH_S,
    device=0 if torch.cuda.is_available() else -1
)

print("Wav2Vec 2.0 pipeline ready on:", device)

## 4. Text normalization

The same normalization procedure is applied to reference and hypothesis text: lowercase, punctuation/non-word replacement, whitespace collapsing, and trimming. Digits are retained.

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

## 5. Resume support

A checkpoint CSV is written after each successful speaker evaluation. On restart, speakers already present in the checkpoint are skipped.

In [ ]:
RESULT_COLUMNS = [
    "speaker", "duration_sec", "reference_words", "hypothesis_words",
    "WER", "CER", "substitutions", "deletions", "insertions", "hits"
]

if CHECKPOINT_PATH.exists():
    results_df = pd.read_csv(CHECKPOINT_PATH)
    completed_speakers = set(results_df["speaker"].astype(str))
    print(f"Existing checkpoint: {len(completed_speakers)} speakers")
else:
    results_df = pd.DataFrame(columns=RESULT_COLUMNS)
    completed_speakers = set()
    print("No checkpoint found. Starting from speaker01.")

## 6. Evaluate all 99 speakers

Each speaker's complete WAV recording is loaded as mono audio at 16 kHz. The ASR pipeline then performs chunked inference and returns the transcription text.

In [ ]:
for speaker_num in range(1, 100):
    speaker_id = f"speaker{speaker_num:02d}"

    if speaker_id in completed_speakers:
        print(f"[SKIP] {speaker_id} already evaluated.")
        continue

    audio_path = DATASET_ROOT / f"{speaker_id}.wav"
    transcript_path = DATASET_ROOT / f"transcript{speaker_num:02d}.txt"

    if not audio_path.is_file():
        print(f"[MISSING AUDIO] {audio_path}")
        continue
    if not transcript_path.is_file():
        print(f"[MISSING TRANSCRIPT] {transcript_path}")
        continue

    start_time = time.time()
    print(f"\nEvaluating {speaker_id} ...")

    try:
        reference = transcript_path.read_text(encoding="utf-8")
        reference_norm = normalize_text(reference)

        audio, sr = librosa.load(
            str(audio_path),
            sr=SAMPLE_RATE,
            mono=True
        )
        duration_sec = len(audio) / SAMPLE_RATE

        result = asr_pipeline(
            audio,
            return_timestamps="char"
        )
        hypothesis = result["text"]
        hypothesis_norm = normalize_text(hypothesis)

        alignment = process_words(reference_norm, hypothesis_norm)
        speaker_wer = wer(reference_norm, hypothesis_norm)
        speaker_cer = cer(reference_norm, hypothesis_norm)

        row = {
            "speaker": speaker_id,
            "duration_sec": duration_sec,
            "reference_words": len(reference_norm.split()),
            "hypothesis_words": len(hypothesis_norm.split()),
            "WER": speaker_wer * 100,
            "CER": speaker_cer * 100,
            "substitutions": alignment.substitutions,
            "deletions": alignment.deletions,
            "insertions": alignment.insertions,
            "hits": alignment.hits
        }

        results_df = pd.concat(
            [results_df, pd.DataFrame([row])],
            ignore_index=True
        )
        results_df.to_csv(CHECKPOINT_PATH, index=False)
        completed_speakers.add(speaker_id)

        print(
            f"{speaker_id}: WER={row['WER']:.2f}% | "
            f"CER={row['CER']:.2f}% | "
            f"elapsed={(time.time() - start_time)/60:.2f} min"
        )

    except Exception as exc:
        print(f"[ERROR] {speaker_id}: {type(exc).__name__}: {exc}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    del audio, result, hypothesis, reference
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nEvaluation loop finished.")
print("Checkpoint saved at:", CHECKPOINT_PATH)

## 7. Speaker-level summary

The mean, median, and standard deviation are calculated across the available speaker-level WER and CER values. These statistics are distinct from aggregate corpus-level WER.

In [ ]:
results_df = pd.read_csv(CHECKPOINT_PATH)
results_df = results_df.sort_values("speaker").reset_index(drop=True)

print("Speakers evaluated:", len(results_df))
display(results_df)

summary = results_df[["WER", "CER"]].agg(["mean", "median", "std"]).T
display(summary)

## 8. Aggregate word-error breakdown

Aggregate WER is calculated from the total number of substitutions, deletions, and insertions divided by the total number of reference words. It is not the same statistic as the mean speaker-level WER.

In [ ]:
total_ref = int(results_df["reference_words"].sum())
total_hits = int(results_df["hits"].sum())
total_S = int(results_df["substitutions"].sum())
total_D = int(results_df["deletions"].sum())
total_I = int(results_df["insertions"].sum())
total_errors = total_S + total_D + total_I

aggregate_wer = total_errors / total_ref if total_ref else float("nan")

aggregate_breakdown = pd.DataFrame({
    "Metric": ["Reference words", "Hits", "Substitutions", "Deletions",
               "Insertions", "Total errors", "Aggregate WER (%)"],
    "Value": [total_ref, total_hits, total_S, total_D, total_I,
              total_errors, aggregate_wer * 100]
})
display(aggregate_breakdown)

if total_ref:
    print(f"Substitution rate: {total_S / total_ref * 100:.2f}%")
    print(f"Deletion rate:     {total_D / total_ref * 100:.2f}%")
    print(f"Insertion rate:    {total_I / total_ref * 100:.2f}%")

if total_errors:
    print(f"Substitution share of errors: {total_S / total_errors * 100:.2f}%")
    print(f"Deletion share of errors:     {total_D / total_errors * 100:.2f}%")
    print(f"Insertion share of errors:    {total_I / total_errors * 100:.2f}%")

## 9. Export results

In [ ]:
PER_SPEAKER_EXPORT = OUTPUT_DIR / "wav2vec2_speaker_level_results.csv"
SUMMARY_EXPORT = OUTPUT_DIR / "wav2vec2_summary.csv"

results_df.to_csv(PER_SPEAKER_EXPORT, index=False)
summary.to_csv(SUMMARY_EXPORT)

print("Saved:", PER_SPEAKER_EXPORT)
print("Saved:", SUMMARY_EXPORT)